# 01 — Load Medellín pollutant network series (PM2.5, PM10, NO2, NO, O3 MDA8)

**Medellín pipeline, mirroring Bogotá's** (for both PM2.5 and O3, per user request).

Unlike Bogotá -- where the raw SISAIRE files are per-station and notebook 02 builds a
network mean from qualifying stations -- for Medellín we start from
`Data/COLOMBIA/Hourmean/<POLLUTANT>_hourmean.{csv,xlsx}`, which already contain an
**hourly city-network-mean** `Medellin` column (built by someone upstream from Medellín's
SIATA/AMVA monitoring stations -- same underlying network the user's 2023 paper used).
This is simpler than Bogotá's per-station approach but means there's no redundancy across
stations to exploit for a completeness-based station-selection step -- QC here is: range
check + Chauvenet's criterion on the hourly network-mean series itself, then aggregate to
daily (plain mean for PM2.5/PM10/NO2/NO, **MDA8** for O3 -- see the Bogotá pipeline's
notebook 04c for why MDA8, not a plain daily mean, is the right O3 metric).

**Coverage**: PM2.5 and NO2/NO run 2015-2022; O3 and PM10 only start 2019 (checked
directly from the source files). The O3 target and anything needing PMC (=PM10-PM2.5)
will therefore be limited to 2019-2022 even though PM2.5-only analysis could use
2015-2022.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd

from common.qc import chauvenet_outlier_mask, PHYSICAL_RANGES

HOURMEAN_DIR = PROJECT_ROOT / "Data" / "COLOMBIA" / "Hourmean"
OUT_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "medellin"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANGES = {"pm25": (0.0, 1000.0), "pm10": (0.0, 1500.0), "no2": (0.0, 500.0),
          "no": (0.0, 500.0), "o3": (0.0, 500.0)}

SOURCES = {
    "pm25": (HOURMEAN_DIR / "PM25_hourmean.xlsx", "xlsx"),
    "pm10": (HOURMEAN_DIR / "PM10_hourmean.xlsx", "xlsx"),
    "no2": (HOURMEAN_DIR / "NO2_hourmean.xlsx", "xlsx"),
    "no": (HOURMEAN_DIR / "NO_hourmean.csv", "csv"),
    "o3": (HOURMEAN_DIR / "O3_hourmean.xlsx", "xlsx"),
}

def load_hourly(path, kind):
    df = pd.read_excel(path) if kind == "xlsx" else pd.read_csv(path)
    df["Dates"] = pd.to_datetime(df["Dates"])
    return df.set_index("Dates")["Medellin"]

hourly = {}
for var, (path, kind) in SOURCES.items():
    s = load_hourly(path, kind)
    s = pd.to_numeric(s, errors="coerce")
    print(f"{var}: {s.notna().sum()} non-null hourly obs, {s.index.min()} to {s.index.max()}")
    hourly[var] = s


pm25: 69383 non-null hourly obs, 2014-01-01 00:00:00 to 2022-12-31 00:00:00


pm10: 34230 non-null hourly obs, 2013-01-01 00:00:00 to 2022-12-31 00:00:00


no2: 68818 non-null hourly obs, 2015-01-01 00:00:00 to 2022-12-31 00:00:00
no: 68599 non-null hourly obs, 2015-01-01 00:00:00 to 2022-12-31 00:00:00


o3: 34235 non-null hourly obs, 2014-01-01 00:00:00 to 2022-12-31 00:00:00


## QC: physical range check + Chauvenet's criterion (on the hourly network-mean series)

In [2]:
for var, s in hourly.items():
    lo, hi = RANGES[var]
    bad = (s < lo) | (s > hi)
    if bad.any():
        print(f"[range check] {var}: {int(bad.sum())} values outside [{lo}, {hi}] -> NaN")
        s[bad] = np.nan
    mask = chauvenet_outlier_mask(s)
    if mask.any():
        print(f"[Chauvenet] {var}: flagged {int(mask.sum())} hourly outliers")
        s[mask] = np.nan
    hourly[var] = s
    print(f"{var}: {s.notna().sum()} non-null hourly obs after QC")


[Chauvenet] pm25: flagged 305 hourly outliers
pm25: 69078 non-null hourly obs after QC
[Chauvenet] pm10: flagged 56 hourly outliers
pm10: 34174 non-null hourly obs after QC
[Chauvenet] no2: flagged 256 hourly outliers
no2: 68562 non-null hourly obs after QC
[Chauvenet] no: flagged 291 hourly outliers
no: 68308 non-null hourly obs after QC
[Chauvenet] o3: flagged 62 hourly outliers
o3: 34173 non-null hourly obs after QC


## Aggregate to daily

Plain daily mean (>=18/24 valid hours) for PM2.5/PM10/NO2/NO; **MDA8** for O3 (8h
rolling mean requiring >=6/8 valid hours, daily max of that rolling mean, kept only if
>=18/24 rolling windows are valid that day) -- same method as Bogotá's notebook 04c.


In [3]:
daily = {}
for var in ["pm25", "pm10", "no2", "no"]:
    s = hourly[var]
    n_valid = s.notna().resample("D").sum()
    daily_mean = s.resample("D").mean()
    daily[var] = daily_mean.where(n_valid >= 18)

s = hourly["o3"]
roll8 = s.rolling(window=8, min_periods=6).mean()
n_valid_per_day = roll8.notna().resample("D").sum()
daily_max = roll8.resample("D").max()
daily["o3"] = daily_max.where(n_valid_per_day >= 18)

network = pd.DataFrame(daily).reset_index().rename(columns={"Dates": "date"})
print(network.shape)
network.describe()


(3652, 6)


,date,pm25,pm10,no2,no,o3
count,3652,2874.000000,1426.000000,2868.000000,2871.000000,1423.000000
mean,2017-12-31 12:00:00,23.102796,39.294625,22.734411,27.412456,55.589910
min,2013-01-01 00:00:00,8.060370,15.357639,4.263838,2.250229,9.682597
25%,2015-07-02 18:00:00,16.948384,31.921094,15.967929,17.028161,46.225258
50%,2017-12-31 12:00:00,21.295833,38.006263,19.336368,24.909722,54.807606
75%,2020-07-01 06:00:00,27.250000,44.859848,25.785627,35.396887,65.075195
max,2022-12-31 00:00:00,64.300000,94.045455,64.571154,94.635233,104.743230
std,NaN,8.638441,11.694230,10.444515,13.825776,14.430096


## Yearly completeness (reported, not filtered -- there's only one network-mean series per pollutant here, no redundant stations to select among)

In [4]:
network["year"] = network["date"].dt.year
for var in ["pm25", "pm10", "no2", "no", "o3"]:
    comp = network.groupby("year")[var].apply(lambda s: s.notna().mean())
    print(f"--- {var} completeness by year ---")
    print(comp.round(2).to_dict())
network = network.drop(columns=["year"])


--- pm25 completeness by year ---
{2013: 0.0, 2014: 0.0, 2015: 0.98, 2016: 0.95, 2017: 1.0, 2018: 1.0, 2019: 1.0, 2020: 0.97, 2021: 1.0, 2022: 0.96}
--- pm10 completeness by year ---
{2013: 0.0, 2014: 0.0, 2015: 0.0, 2016: 0.0, 2017: 0.0, 2018: 0.0, 2019: 1.0, 2020: 1.0, 2021: 0.99, 2022: 0.92}
--- no2 completeness by year ---
{2013: 0.0, 2014: 0.0, 2015: 1.0, 2016: 1.0, 2017: 1.0, 2018: 1.0, 2019: 0.97, 2020: 0.98, 2021: 0.99, 2022: 0.91}
--- no completeness by year ---
{2013: 0.0, 2014: 0.0, 2015: 1.0, 2016: 1.0, 2017: 1.0, 2018: 1.0, 2019: 0.97, 2020: 0.99, 2021: 1.0, 2022: 0.9}
--- o3 completeness by year ---
{2013: 0.0, 2014: 0.0, 2015: 0.0, 2016: 0.0, 2017: 0.0, 2018: 0.0, 2019: 0.99, 2020: 1.0, 2021: 0.99, 2022: 0.92}


## Save

In [5]:
out_path = OUT_DIR / "medellin_network_daily_clean.parquet"
network.to_parquet(out_path, index=False)
print("Saved:", out_path, network.shape)
print("date range:", network['date'].min(), "to", network['date'].max())


Saved: /path/to/project/Data/processed/colombia/medellin/medellin_network_daily_clean.parquet (3652, 6)
date range: 2013-01-01 00:00:00 to 2022-12-31 00:00:00
